# Lab 1: Classification — Will I Pass This Class?

*To run this in Google Colab, upload this notebook file directly to Colab.*

## Learning Objectives
- Understand classification as a supervised learning task that predicts categories instead of numbers
- Train and interpret a Decision Tree classifier on a student-success problem
- Recognize underfitting, good balance, and overfitting as tree depth changes
- Compare one Decision Tree with a stronger Random Forest baseline
- Use a fitted classifier to make a personal "Will I pass?" prediction from your own feature values

## Overview
This lab is intentionally relatable. Instead of classifying penguins or plants, you will work with a synthetic but realistic student-success dataset. The features are simple enough to imagine for yourself: study time, past failures, absences, whether you completed the lab, and average sleep.

## By The End Of This Lab
- Explain what makes a problem classification rather than regression.
- Describe a Decision Tree as a sequence of yes/no questions about the features.
- Use train/test splits and accuracy carefully instead of trusting one number blindly.
- Spot when a shallow tree is too simple and when a deep tree starts memorizing noise.
- Interpret a Random Forest as many trees voting together, then use it to predict your own success profile.


## Local Setup (Optional)
If you are running this notebook on your own machine, create an environment first, register it as a Jupyter kernel, and then select that kernel before running the notebook.

**Conda**
```bash
conda create -n ai101-ml python=3.12 -y
conda activate ai101-ml
python -m pip install numpy pandas matplotlib seaborn scikit-learn jupyter ipykernel
python -m ipykernel install --user --name ai101-ml --display-name "Python (ai101-ml)"
```

**uv**
```bash
uv venv --python 3.12 .venv
source .venv/bin/activate
uv pip install numpy pandas matplotlib seaborn scikit-learn jupyter ipykernel
python -m ipykernel install --user --name ai101-ml-uv --display-name "Python (ai101-ml-uv)"
```

Then launch Jupyter with `jupyter lab` or `jupyter notebook` and choose the kernel you just created.


## Troubleshooting
- If you see `ModuleNotFoundError` for `seaborn`, `sklearn`, `jupyter`, or `ipykernel`, install the missing package inside the same environment that the notebook kernel is using.
- If a cell says a variable such as `df`, `X_train`, `tree`, or `rf` does not exist, you probably ran cells out of order. Restart the kernel and run from the top.
- If you install packages while the notebook is open, restart the kernel before trying again.
- If outputs look stale after you edit code, use `Kernel -> Restart Kernel and Run All Cells`.
- If a plot looks strange after you change manual values near the end, rerun the training cells first so the model objects are fresh.


## How to Use This Lab
- Run the notebook from top to bottom. Most later cells depend on variables created earlier.
- Before each task cell, pause and predict what the plot, metric, or model behavior should look like.
- After each result, ask three questions: What are the axes? What pattern stands out? What would count as a failure?
- Use the hints when you need them, but try to write the core lines yourself before checking the solution notebook.
- If a result surprises you, do not skip past it. Surprise is usually the place where the concept becomes clear.


## Lab Roadmap
| Task | What You Will Build | What To Notice | Main Output |
|---|---|---|---|
| T1 | A realistic student-success dataset and feature matrix | The target depends on several factors at once, not one magic feature | Dataset preview, class balance, and feature definitions |
| T2 | A stratified train/test split | The held-out set should stay fair and keep pass/fail balance | Training/test set sizes and class counts |
| T3 | A first Decision Tree classifier | Trees ask a sequence of yes/no questions until classes become purer | Accuracy, confusion matrix, depth, leaves, and a readable tree plot |
| T4 | A depth sweep for one tree | Shallow trees underfit and overly deep trees can memorize | Train/test accuracy by depth plus example trees |
| T5 | A Random Forest comparison | Many trees voting together are usually more stable than one tree | Tree-vs-forest accuracy and feature importances |
| T6 | A "predict your own success" demo | Small changes in the inputs can move the prediction and the confidence | One custom student profile with predicted pass probability |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'legend.fontsize': 11,
})

COLORS = {
    'teal': '#2A9D8F',
    'orange': '#F4A261',
    'red': '#E76F51',
    'navy': '#264653',
    'gold': '#E9C46A',
}


## 1. A Student-Success Dataset
This notebook uses a **teaching dataset**. The rows are synthetic, but the relationships are designed to feel realistic:

- more study time usually helps
- more past failures usually hurts
- more absences usually hurts
- completing the lab often helps
- sleeping far too little usually hurts

The important point is not that this is a perfect portrait of real students. The point is that it is realistic enough to be relatable, while still clear enough to teach what the model is doing.

### What The Features Mean
| Feature | Plain-English meaning |
|---|---|
| `study_time_hours` | Average study hours per week |
| `past_failures` | How many related course failures the student already has |
| `absences` | Number of missed class sessions |
| `did_lab` | `1` if the student completed the lab work, `0` otherwise |
| `sleep_hours` | Average hours of sleep per night |
| `pass_class` | `1` means predicted to pass, `0` means needs more support |


In [ ]:
def make_student_success_data(n_students=320, random_state=42):
    rng = np.random.default_rng(random_state)

    study_time_hours = np.clip(rng.normal(loc=6.2, scale=2.3, size=n_students), 0.5, 14.0)
    past_failures = rng.choice([0, 1, 2, 3], size=n_students, p=[0.58, 0.24, 0.12, 0.06])

    extra_absence_burst = rng.binomial(1, 0.18, size=n_students) * rng.integers(2, 8, size=n_students)
    absences = np.clip(rng.poisson(lam=3.5, size=n_students) + extra_absence_burst, 0, 25)

    did_lab_prob = np.clip(0.34 + 0.045 * study_time_hours - 0.10 * past_failures, 0.12, 0.95)
    did_lab = rng.binomial(1, did_lab_prob)

    sleep_hours = np.clip(
        rng.normal(loc=6.9, scale=1.0, size=n_students) - 0.18 * past_failures + 0.18 * did_lab,
        4.0,
        9.5,
    )

    score = (
        -1.85
        + 0.47 * study_time_hours
        - 1.15 * past_failures
        - 0.08 * absences
        + 1.05 * did_lab
        - 0.30 * (sleep_hours - 7.1) ** 2
        + rng.normal(loc=0.0, scale=0.85, size=n_students)
    )

    pass_prob = 1 / (1 + np.exp(-score))
    pass_class = rng.binomial(1, pass_prob)

    df = pd.DataFrame({
        'study_time_hours': np.round(study_time_hours, 1),
        'past_failures': past_failures,
        'absences': absences,
        'did_lab': did_lab,
        'sleep_hours': np.round(sleep_hours, 1),
        'pass_class': pass_class,
    })
    df['pass_label'] = df['pass_class'].map({0: 'Need support', 1: 'Pass'})
    df['did_lab_label'] = df['did_lab'].map({0: 'No', 1: 'Yes'})
    return df


### T1 — Build The Student-Success Dataset

**Goal:** Generate the teaching dataset, choose the feature columns, and separate the target.

**Starter variables:** `make_student_success_data`

**You will implement:** the DataFrame `df`, the feature list `feature_cols`, the target name `target_col`, and then `X` and `y`.

**Hints:** Call the generator with no arguments first. Keep the feature order exactly the same as in the table above.

**Check yourself:** The dataset should have six core columns plus helper label columns, and the pass/fail counts should not be wildly imbalanced.


In [ ]:
## T1 TODO: Create the teaching dataset and define the feature / target columns ##
# 1. Call make_student_success_data() and store the result in df.
# 2. Create feature_cols with the five feature names in the order listed above.
# 3. Set target_col = 'pass_class', then create X and y from df.
# 4. Print the dataset shape and inspect the class counts.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sns.countplot(
    data=df,
    x='pass_label',
    hue='pass_label',
    palette=[COLORS['orange'], COLORS['teal']],
    ax=axes[0],
    legend=False,
)
axes[0].set_title('How Balanced Is Pass vs Need Support?')
axes[0].set_xlabel('Outcome')
axes[0].set_ylabel('Number of students')

sns.scatterplot(
    data=df,
    x='study_time_hours',
    y='absences',
    hue='pass_label',
    style='did_lab_label',
    palette={'Need support': COLORS['red'], 'Pass': COLORS['teal']},
    alpha=0.78,
    ax=axes[1],
)
axes[1].set_title('Study Time vs Absences')
axes[1].set_xlabel('Study time per week (hours)')
axes[1].set_ylabel('Absences')


In [ ]:
fig = plt.figure(figsize=(16.5, 9.5), constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.18], hspace=0.16, wspace=0.12)

outcome_order = ['Pass', 'Need support']
palette = {'Pass': COLORS['orange'], 'Need support': COLORS['teal']}

failure_summary = (
    df.groupby('past_failures')['pass_class']
    .agg(pass_rate='mean', count='size')
    .reset_index()
    .sort_values('past_failures')
)
failure_summary['pass_rate_pct'] = 100 * failure_summary['pass_rate']

ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(
    failure_summary['past_failures'].astype(str),
    failure_summary['pass_rate_pct'],
    color=COLORS['orange'],
    edgecolor='white',
    linewidth=1.8,
    width=0.68,
)
ax1.axhline(50, color='#6c757d', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.set_title('Pass Rate Falls Sharply After Earlier Failures', pad=10)
ax1.set_xlabel('Number of past failures')
ax1.set_ylabel('Students who pass (%)')
ax1.set_ylim(0, 100)
for bar, pct, count in zip(bars, failure_summary['pass_rate_pct'], failure_summary['count']):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f'{pct:.0f}%\n(n={count})',
        ha='center',
        va='bottom',
        fontsize=10,
        color='#36454F',
    )
ax1.text(
    0.98,
    0.08,
    'Dashed line = coin-flip level',
    transform=ax1.transAxes,
    ha='right',
    va='bottom',
    fontsize=10,
    color='#6c757d',
)

ax2 = fig.add_subplot(gs[0, 1])
sns.violinplot(
    data=df,
    x='pass_label',
    y='sleep_hours',
    order=outcome_order,
    hue='pass_label',
    palette=palette,
    inner=None,
    cut=0,
    linewidth=1.3,
    saturation=0.95,
    ax=ax2,
    legend=False,
)
sns.boxplot(
    data=df,
    x='pass_label',
    y='sleep_hours',
    order=outcome_order,
    width=0.22,
    showcaps=False,
    showfliers=False,
    boxprops={'facecolor': 'white', 'alpha': 0.95, 'edgecolor': '#333333'},
    whiskerprops={'color': '#333333'},
    medianprops={'color': '#333333', 'linewidth': 2},
    ax=ax2,
)
sns.stripplot(
    data=df.sample(min(len(df), 140), random_state=42),
    x='pass_label',
    y='sleep_hours',
    order=outcome_order,
    color='#1f2937',
    alpha=0.16,
    size=3,
    jitter=0.16,
    ax=ax2,
)
ax2.set_title('Sleep Helps a Bit, But It Is Not the Main Driver', pad=10)
ax2.set_xlabel('Outcome')
ax2.set_ylabel('Sleep per night (hours)')
ax2.set_ylim(4, 9.7)

corr = df[['study_time_hours', 'past_failures', 'absences', 'did_lab', 'sleep_hours', 'pass_class']].corr(numeric_only=True)
corr_display = corr.rename(
    index={
        'study_time_hours': 'Study',
        'past_failures': 'Failures',
        'absences': 'Absences',
        'did_lab': 'Did lab',
        'sleep_hours': 'Sleep',
        'pass_class': 'Pass',
    },
    columns={
        'study_time_hours': 'Study',
        'past_failures': 'Failures',
        'absences': 'Absences',
        'did_lab': 'Did lab',
        'sleep_hours': 'Sleep',
        'pass_class': 'Pass',
    },
)

ax3 = fig.add_subplot(gs[1, :])
sns.heatmap(
    corr_display,
    annot=True,
    cmap='crest',
    fmt='.2f',
    vmin=-0.4,
    vmax=1.0,
    linewidths=1,
    linecolor='white',
    square=False,
    cbar_kws={'shrink': 0.82, 'label': 'Correlation'},
    annot_kws={'size': 11},
    ax=ax3,
)
ax3.set_title('Correlation Snapshot: Study Time Helps, Failures and Skipping Labs Hurt', pad=12)
ax3.tick_params(axis='x', rotation=18)
ax3.tick_params(axis='y', rotation=0)
for label in ax3.get_xticklabels():
    label.set_horizontalalignment('right')


**What you should notice**

- Students who study more and miss fewer classes tend to sit closer to the "Pass" region.
- Past failures are a strong warning sign, but they are not the whole story by themselves.
- `did_lab` helps, but it does not magically override every other feature.

**Common beginner mistake**

- Looking at one plot and deciding one feature "causes" the outcome. These features work together.


## 2. Train/Test Split
Before training any classifier, we need a fair final exam.

- **Training set:** the rows the model is allowed to learn from
- **Test set:** the rows we hide until the end

Because this is a pass/fail problem, we also want to keep the class balance similar across both splits. That is why `stratify=y` matters.


### T2 — Create A Fair Train/Test Split

**Goal:** Hold out 20% of the students for evaluation while keeping the pass/fail balance fair.

**Starter variables:** `X`, `y`

**You will implement:** one call to `train_test_split(...)` that returns `X_train`, `X_test`, `y_train`, and `y_test`.

**Hints:** Use `test_size=0.2`, `random_state=42`, and `stratify=y`.

**Check yourself:** The training set should be larger than the test set, and the pass/fail ratio should stay similar in both splits.


In [ ]:
## T2 TODO: Create a stratified train/test split ##
# Use train_test_split(...) with test_size=0.2, random_state=42, and stratify=y.
# Store the results in X_train, X_test, y_train, and y_test.
# Then print the shapes plus the class balance for both splits.


**What you should notice**

- The train set is larger than the test set.
- The pass/fail proportions should look similar in both splits.

**What this does not mean**

- A fair split does not guarantee a good model. It only makes the evaluation more trustworthy.


## 3. Decision Tree Classifier
A Decision Tree keeps asking yes/no questions such as:

- "Did this student complete the lab?"
- "Are absences above a threshold?"
- "Is study time high enough?"

The tree keeps splitting until the groups become purer. That makes trees easy to explain, but it also means they can become too detailed if we let them grow without enough restraint.


### T3 — Train And Evaluate A Decision Tree

**Goal:** Build the first classifier and inspect how complex it becomes.

**Starter variables:** `X_train`, `X_test`, `y_train`, `y_test`

**You will implement:** a `DecisionTreeClassifier`, its fit on the training data, predictions on the test set, and the accuracy score.

**Hints:** Start with `DecisionTreeClassifier(random_state=42)`. After fitting, inspect both `tree.get_depth()` and `tree.get_n_leaves()`.

**Check yourself:** The notebook should print a sensible test accuracy plus a nontrivial depth and number of leaves.


In [ ]:
## T3 TODO: Train a Decision Tree and evaluate it ##
# 1. Create a DecisionTreeClassifier(random_state=42).
# 2. Fit it on X_train and y_train.
# 3. Predict on X_test, store the predictions, and compute the accuracy.
# 4. Print the accuracy, tree depth, number of leaves, and a classification report.


In [ ]:
cm = confusion_matrix(y_test, tree_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Need support', 'Actual: Pass'],
    columns=['Predicted: Need support', 'Predicted: Pass'],
)

fig, ax = plt.subplots(figsize=(7.2, 6.2), constrained_layout=True)
sns.heatmap(
    cm_df,
    annot=True,
    fmt='d',
    cmap='crest',
    cbar=False,
    square=True,
    linewidths=1.5,
    linecolor='white',
    annot_kws={'size': 22},
    ax=ax,
)
ax.set_title('Decision Tree Confusion Matrix', pad=10)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)


**How to interpret the printed numbers**

- Accuracy tells you the overall fraction of correct predictions.
- The confusion matrix tells you *which kind* of mistake the model makes.
- Tree depth and leaf count tell you how complicated the model became.

**What to look for in the next tree plot**

- The first few splits tell you which features the model trusts most.
- The plot is intentionally truncated so the main decision path stays readable.

**Common beginner mistake**

- Celebrating a decent accuracy score without checking whether the tree grew surprisingly large or makes one-sided mistakes.


In [ ]:
tree_plot_depth = 2

fig, ax = plt.subplots(figsize=(20, 8), constrained_layout=True)
plot_tree(
    tree,
    feature_names=feature_cols,
    class_names=['Need support', 'Pass'],
    filled=True,
    rounded=True,
    impurity=False,
    proportion=True,
    max_depth=tree_plot_depth,
    precision=2,
    fontsize=11,
    ax=ax,
)
ax.set_title('Decision Tree Structure (Top 3 Levels Only)', pad=14)
ax.text(
    0.5,
    -0.06,
    f'Full tree depth = {tree.get_depth()} with {tree.get_n_leaves()} leaves. '
    'The plot is intentionally truncated so the big-picture logic stays readable.',
    transform=ax.transAxes,
    ha='center',
    va='top',
    fontsize=11,
)


## 4. Model Complexity: When Trees Underfit Or Overfit
A shallow tree may be too simple to capture the pattern. A very deep tree may memorize tiny quirks of the training set instead of learning the real signal.

That is why we track **both** training accuracy and test accuracy. The gap between them tells you whether the model is learning a stable pattern or just memorizing details.


### T4 — Measure Underfitting, Good Balance, And Overfitting

**Goal:** Compare train accuracy and test accuracy as you change `max_depth`.

**Starter variables:** `depths`, `train_accs`, `test_accs`, and the train/test split variables

**You will implement:** a loop that trains one Decision Tree per depth and stores both train and test accuracy.

**Hints:** Create a fresh `DecisionTreeClassifier(max_depth=d, random_state=42)` inside the loop. Watch the train/test gap, not just the best single point.

**Check yourself:** Very shallow depth should look weaker on both train and test, while deep trees should drive training accuracy higher than test accuracy.


In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, None]
train_accs, test_accs, leaf_counts = [], [], []

## T4 TODO: Train one Decision Tree per depth and store train/test accuracy ##
# Inside a loop over depths:
# 1. Create DecisionTreeClassifier(max_depth=depth, random_state=42)
# 2. Fit it on the training data
# 3. Append the train accuracy, test accuracy, and number of leaves


In [ ]:
depth_labels = ['1', '2', '3', '4', '5', '6', '8', 'None']
x_pos = np.arange(len(depths))
best_depth_idx = int(np.argmax(test_accs))

fig, ax = plt.subplots(figsize=(10, 5.6), constrained_layout=True)
ax.plot(x_pos, train_accs, 'o-', linewidth=2.5, markersize=8, color=COLORS['orange'], label='Train accuracy')
ax.plot(x_pos, test_accs, 'o-', linewidth=2.5, markersize=8, color=COLORS['teal'], label='Test accuracy')
ax.set_xticks(x_pos)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree Accuracy vs Tree Depth')
ax.set_ylim(0.55, 1.03)
ax.grid(True, alpha=0.25)
ax.legend()

note_box = dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.92, edgecolor='none')
ax.annotate(
    'Underfit risk',
    xy=(x_pos[0], test_accs[0]),
    xytext=(18, -34),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='black', shrinkA=0, shrinkB=4),
    ha='left',
    va='top',
    bbox=note_box,
)
ax.annotate(
    f'Best test accuracy\n(depth={depth_labels[best_depth_idx]})',
    xy=(x_pos[best_depth_idx], test_accs[best_depth_idx]),
    xytext=(12, 14),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color=COLORS['teal'], shrinkA=0, shrinkB=4),
    color=COLORS['teal'],
    ha='left',
    va='bottom',
    bbox=note_box,
)
ax.annotate(
    'Overfit risk',
    xy=(x_pos[-1], test_accs[-1]),
    xytext=(-24, -36),
    textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='black', shrinkA=0, shrinkB=4),
    ha='right',
    va='top',
    bbox=note_box,
)


In [ ]:
example_depths = [(1, 'Underfit example'), (3, 'Good balance example'), (8, 'Overfit risk example')]
fig, axes = plt.subplots(1, 3, figsize=(20, 6), constrained_layout=True)

for ax, (depth, title) in zip(axes, example_depths):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    plot_tree(
        model,
        feature_names=feature_cols,
        class_names=['Need support', 'Pass'],
        filled=True,
        rounded=True,
        impurity=False,
        max_depth=2,
        fontsize=8,
        ax=ax,
    )
    ax.set_title(title)


**What you should notice**

- Shallow trees miss useful structure and underfit.
- A middle depth often gives the best balance between fit and generalization.
- Deep trees can drive training accuracy very high while test accuracy stops improving much.

**What this does not mean**

- The deepest tree is not automatically "best" just because it learned more details.


## 5. Random Forest: A Stronger Team Of Trees
A Random Forest trains many different trees, each on a slightly different view of the data, and lets them vote.

That usually makes the final model more stable than one single tree. The tradeoff is that the forest is harder to explain in full detail, because you are now reasoning about a crowd rather than one path of questions.


### T5 — Train And Evaluate A Random Forest

**Goal:** Compare one tree with a whole team of trees voting together.

**Starter variables:** `X_train`, `X_test`, `y_train`, `y_test`

**You will implement:** a `RandomForestClassifier`, its predictions, and the resulting test accuracy.

**Hints:** Use `RandomForestClassifier(n_estimators=200, random_state=42)`. The workflow is intentionally similar to the Decision Tree API: create, fit, predict, score.

**Check yourself:** The forest should usually match or beat the single-tree baseline while feeling less fragile.


In [ ]:
## T5 TODO: Train a Random Forest and compare it with the Decision Tree ##
# 1. Create RandomForestClassifier(n_estimators=200, random_state=42)
# 2. Fit it on the training data
# 3. Predict on X_test and compute the Random Forest accuracy
# 4. Compare it with the Decision Tree accuracy stored in tree_acc


In [ ]:
compare_df = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [tree_acc, rf_acc],
})
importance_df = (
    pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
sns.barplot(data=compare_df, x='Model', y='Accuracy', hue='Model', palette=[COLORS['orange'], COLORS['teal']], legend=False, ax=axes[0])
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title('Decision Tree vs Random Forest')

sns.barplot(data=importance_df, x='importance', y='feature', hue='feature', palette='crest', legend=False, ax=axes[1])
axes[1].set_title('Random Forest Feature Importance')
axes[1].set_xlabel('Importance')
axes[1].set_ylabel('')


**What you should notice**

- The Random Forest usually improves stability by averaging many trees together.
- The feature-importance chart should roughly match the patterns you already saw in the plots.

**Common beginner mistake**

- Assuming the forest is "better" in every way. It is often more accurate, but much less interpretable than one tree.


## 6. Predict Your Own Success
This is the fun part: you can now feed the model your own feature values and see how it responds.

Important caveat: this is a teaching exercise, not life advice. The prediction only reflects the synthetic pattern used to generate this notebook's data.


### T6 — Predict Your Own Success

**Goal:** Create one custom student profile and ask the Random Forest for a pass probability.

**Starter variables:** `rf`, `feature_cols`, and the example dictionary below

**You will implement:** a one-row DataFrame from `my_profile`, then the Random Forest prediction plus predicted probability.

**Hints:** `pd.DataFrame([my_profile])` is the easiest way to create a single-row feature table. Use `predict_proba(...)` if you want a confidence-like score.

**Check yourself:** Changing the values inside `my_profile` should change both the class prediction and the pass probability.


In [ ]:
my_profile = {
    'study_time_hours': 7.5,
    'past_failures': 0,
    'absences': 2,
    'did_lab': 1,
    'sleep_hours': 7.2,
}

## T6 TODO: Turn my_profile into a one-row DataFrame and ask rf for a prediction ##
# 1. Build my_profile_df from the dictionary above
# 2. Reorder the columns with feature_cols
# 3. Compute the predicted class and pass probability from rf
# 4. Print the result and try editing the values to see how the output changes


**What you should notice**

- This model never learned one giant formula. It learned decision rules from data.
- `did_lab`, absences, and past failures often matter early because they create strong splits.
- The Random Forest is usually steadier than one tree, but it is less transparent.

**What this does not mean**

- The prediction is not destiny.
- A model trained on synthetic data is useful for learning concepts, not for making real academic decisions.

**Common beginner mistake**

- Treating one probability as truth instead of asking what data generated the model and whether the features are even realistic.
